# Aula 11 — Técnicas Complementares
## Multimodal RAG · Compressão · ColBERT · Time-Aware · DSPy

**Curso:** MBA em RAG & CAG Aplicados a Direito e Segurança Pública  
**Aula:** 11 de 12 · Carga Horária: 5h  
**Proporção:** 30% Teoria · 70% Prática  
**Nível:** Complementar / Avançado

---

## Ementa

Esta aula apresenta um panorama de técnicas complementares ao pipeline RAG canônico, abordando cinco frentes avançadas:

1. **Multimodal RAG** — recuperação de texto, imagens e tabelas em espaço unificado
2. **Compression RAG / LLMLingua** — redução de contexto via token pruning
3. **ColBERT / Late Interaction** — retrieval granular token a token
4. **Time-Aware RAG** — decay temporal para relevância cronológica
5. **DSPy** — otimização automática de pipelines RAG

---

## Objetivos de Aprendizagem

Ao concluir esta aula, o aluno será capaz de:

- Implementar Multimodal RAG indexando texto e imagens extraídas pelo Docling
- Aplicar LLMLingua para compressão de chunks e medir impacto em custo e qualidade
- Comparar ColBERT (late interaction) com bi-encoder em corpus técnico denso
- Configurar decay temporal no OpenSearch para Time-Aware RAG
- Usar DSPy para otimizar automaticamente prompts de um pipeline RAG

---

## Referências Bibliográficas (ABNT)

> JIANG, Huiqiang et al. **LLMLingua: Compressing Prompts for Accelerated Inference of Large Language Models**. In: *Proceedings of EMNLP 2023*. Singapore: ACL, 2023. Disponível em: https://arxiv.org/abs/2310.05736
>
> KHATTAB, Omar; ZAHARIA, Matei. **ColBERT: Efficient and Effective Passage Search via Contextualized Late Interaction over BERT**. In: *Proceedings of SIGIR 2020*. New York: ACM, 2020. Disponível em: https://arxiv.org/abs/2004.12832
>
> KHATTAB, Omar et al. **DSPy: Compiling Declarative Language Model Calls into Self-Improving Pipelines**. In: *ICLR 2024*. Vienna: OpenReview, 2024. Disponível em: https://arxiv.org/abs/2310.03714
>
> YASUNAGA, Michihiro et al. **Retrieval-Augmented Multimodal Language Modeling**. In: *NeurIPS 2023*. New Orleans: NeurIPS Foundation, 2023.
>
> WAN, Zilong et al. **Speculative RAG: Enhancing Retrieval Augmented Generation through Drafting**. *arXiv:2407.08223*, Google Research, 2024.

---
# PARTE I — TEORIA (30%)
---

## 1. Multimodal RAG — Texto, Imagens e Tabelas Unificados

### 1.1 Motivação

A maioria dos sistemas RAG trata exclusivamente texto. No entanto, documentos jurídicos, laudos policiais, relatórios forenses e legislações frequentemente contêm **tabelas, gráficos, plantas e imagens** que carregam informação crítica. Um sistema que ignora essas modalidades é estruturalmente incompleto para o domínio de Direito e Segurança Pública.

**Exemplos práticos no domínio:**
- Boletins de ocorrência com fotos de evidências digitalizadas
- Laudos periciais com tabelas de análise toxicológica
- Mapas de calor criminal em apresentações de PowerPoint
- Organogramas de facções em documentos de inteligência

### 1.2 Arquitetura Multimodal RAG

```
┌─────────────────────────────────────────────────────────────┐
│                    INGESTÃO MULTIMODAL                       │
│                                                              │
│  PDF/DOCX ──► Docling ──► ┌─ Texto (parágrafos, títulos)  │
│                            ├─ Tabelas (markdown/HTML)       │
│                            └─ Imagens (PNG extraídas)       │
│                                      │                       │
│              ┌───────────────────────┤                       │
│              ▼                       ▼                       │
│         BGE-M3                    CLIP/ImageBind             │
│    (texto + tabelas)            (imagens)                    │
│              │                       │                       │
│              └───────┬───────────────┘                       │
│                      ▼                                       │
│              OpenSearch (índice unificado)                   │
└─────────────────────────────────────────────────────────────┘
                         │
              ┌──────────▼──────────┐
              │     QUERY TIME      │
              │  Query ──► Embedder │
              │  KNN Search         │
              │  Retorna texto      │
              │  + imagens          │
              │  + tabelas          │
              └─────────────────────┘
```

### 1.3 Docling — Extração Inteligente de PDFs

O **Docling** (IBM Research, 2024) é uma biblioteca Python especializada na conversão de documentos complexos (PDF, DOCX, PPTX) para formatos estruturados. Suas principais capacidades:

| Capacidade | Descrição |
|---|---|
| Layout Analysis | Detecta colunas, headers, footers, notas de rodapé |
| Table Extraction | Extrai tabelas para Markdown ou DataFrame Pandas |
| Image Extraction | Salva imagens incorporadas com metadados de posição |
| OCR Integrado | Processa documentos digitalizados |
| Chunking Semântico | Divide por seção lógica, não por tamanho fixo |

### 1.4 CLIP — Embeddings de Imagem

O **CLIP** (Contrastive Language-Image Pre-Training, OpenAI 2021) projeta texto e imagens no mesmo espaço vetorial de 512 dimensões. Isso permite:

- Buscar imagens com queries textuais: *"fluxograma de processo licitatório"*
- Comparar similaridade entre imagem e texto diretamente
- Indexar imagens no mesmo índice que documentos textuais

**Fórmula de similaridade:**
```
score(query_text, image) = cosine_similarity(
    CLIP_text_encoder(query),
    CLIP_image_encoder(image)
)
```

### 1.5 Ficha Técnica #T17

| Campo | Conteúdo |
|---|---|
| **ID** | #T17 — Multimodal RAG |
| **Categoria** | Complementar |
| **Stack** | Docling + CLIP + BGE-M3 + OpenSearch |
| **Vantagem principal** | Cobre lacunas onde informação essencial está em imagens/tabelas |
| **Limitação principal** | Infraestrutura complexa; embeddings multimodais ainda em evolução |
| **Referência** | Yasunaga et al., NeurIPS 2023 |

---
## 2. Compression RAG / LLMLingua — Redução de Contexto por Token Pruning

### 2.1 O Problema do Contexto Extenso

Pipelines RAG ingênuos concatenam todos os chunks recuperados e os enviam diretamente ao LLM. Isso gera três problemas:

1. **Custo elevado:** Modelos como GPT-4 cobram por token de entrada. 10 chunks × 512 tokens = 5.120 tokens de contexto por query.
2. **Latência:** Contextos longos aumentam o tempo de inferência.
3. **Lost in the Middle:** LLMs tendem a ignorar informações no meio de contextos longos (Liu et al., 2023).

### 2.2 LLMLingua — Como Funciona

O **LLMLingua** (Microsoft Research, 2023) resolve esse problema com **perplexity-based token pruning**:

```
PIPELINE:

Chunks recuperados
  │
  ▼
Small LM (GPT-2 ou Llama-7B)
  │   Calcula perplexidade de cada token
  │   dado o contexto anterior
  ▼
Token Scoring:
  • Alta perplexidade = token surpreendente = potencialmente importante
  • Baixa perplexidade = token previsível = candidato a remoção
  │
  ▼
Budget Controller:
  • Define taxa de compressão (ex: 70%)
  • Remove tokens de baixa perplexidade
  │
  ▼
Contexto comprimido → LLM grande (GPT-4, Claude)
```

### 2.3 Exemplo de Compressão

**Texto original (52 tokens):**
```
O artigo 5º da Constituição Federal estabelece que todos são iguais 
perante a lei, sem distinção de qualquer natureza, garantindo-se aos 
brasileiros e aos estrangeiros residentes no País a inviolabilidade 
do direito à vida, à liberdade, à igualdade, à segurança e à propriedade.
```

**Texto comprimido LLMLingua (≈16 tokens, taxa 70%):**
```
Art.5 CF: todos iguais lei, garantindo vida, liberdade, igualdade, 
segurança, propriedade.
```

**Análise:** Termos funcionais de alta previsibilidade (*"da"*, *"que"*, *"aos"*, *"e aos"*) são removidos. O núcleo semântico é preservado.

### 2.4 RECOMP — Alternativa Abstractiva

Enquanto LLMLingua faz **compressão extrativa** (remove tokens), o **RECOMP** (Xu et al., 2023) faz **compressão abstractiva**: usa um modelo de sumarização fine-tunado para redigir uma versão condensada do contexto. É mais custoso mas preserva melhor a coerência.

| Método | Tipo | Custo | Qualidade |
|---|---|---|---|
| LLMLingua | Extrativo | Baixo | Alta (para textos técnicos) |
| LLMLingua-2 | Extrativo | Médio | Muito Alta |
| RECOMP Extractive | Extrativo | Médio | Alta |
| RECOMP Abstractive | Abstractivo | Alto | Muito Alta |

### 2.5 Ficha Técnica #T23

| Campo | Conteúdo |
|---|---|
| **ID** | #T23 — Compression RAG / LLMLingua |
| **Categoria** | Complementar |
| **Stack** | LLMLingua + GPT-2 (small LM) + LangFuse |
| **Redução típica** | 2–5× menos tokens de entrada |
| **Vantagem principal** | Reduz custo e latência sem pipeline de reranking separado |
| **Limitação principal** | Risco de perda de informação sutil; modelo adicional no pipeline |
| **Referência** | Jiang et al., EMNLP 2023 |

---
## 3. ColBERT — Late Interaction e Retrieval Granular

### 3.1 Limitações dos Bi-Encoders

Os embeddings convencionais (BGE-M3, E5, text-embedding-3) são **bi-encoders**: comprimem todo o documento em um único vetor. Isso é eficiente mas perde granularidade:

```
Bi-Encoder:
"O suspeito portava arma de fogo" ──► [0.21, -0.43, 0.87, ...] (1 vetor)

Query: "tipo de arma" ──► [0.18, -0.39, 0.91, ...] (1 vetor)

Score = cosine(query_vec, doc_vec) = 0.74
```

O problema: termos irrelevantes do documento *diluem* o vetor, reduzindo a precisão em queries específicas.

### 3.2 ColBERT — Late Interaction com MaxSim

O **ColBERT** (Khattab & Zaharia, 2020) mantém **um vetor por token**, computando a relevância via **MaxSim**:

```
ColBERT:
Documento:
  "O"        ──► v1
  "suspeito" ──► v2
  "portava"  ──► v3
  "arma"     ──► v4
  "de"       ──► v5
  "fogo"     ──► v6

Query:
  "tipo"  ──► q1
  "de"    ──► q2
  "arma"  ──► q3

MaxSim Score:
  Para q1 ("tipo"): max(cos(q1,v1), cos(q1,v2), ..., cos(q1,v6)) = cos(q1,v4) = 0.62
  Para q2 ("de"):   max(cos(q2,v1), ..., cos(q2,v6)) = cos(q2,v5) = 0.91
  Para q3 ("arma"): max(cos(q3,v1), ..., cos(q3,v6)) = cos(q3,v4) = 0.97

Score Final = sum(MaxSim) = 0.62 + 0.91 + 0.97 = 2.50
```

### 3.3 PLAID Engine — Eficiência em Escala

Armazenar vetores por token seria inviável em grande escala. O **PLAID** (Santhanam et al., 2022) resolve isso com:

1. **Compressão de vetores** via clustering (k-means nos embeddings de token)
2. **Centróides compartilhados** — tokens similares apontam para o mesmo centróide
3. **Candidate generation** — filtragem rápida por centróide antes do MaxSim exato

**Resultado:** Precisão de cross-encoder com velocidade próxima a bi-encoder.

### 3.4 Comparativo de Métodos de Retrieval

| Método | nDCG@10 típico | Latência | Índice (tamanho) |
|---|---|---|---|
| BM25 | 0.45–0.55 | ~5ms | Pequeno |
| Bi-Encoder (BGE-M3) | 0.60–0.70 | ~10ms | Médio |
| ColBERT | 0.70–0.80 | ~30ms | Grande (3-5×) |
| Cross-Encoder | 0.75–0.85 | ~200ms | N/A (sem índice) |

### 3.5 RAGatouille — ColBERT Simplificado

A biblioteca **RAGatouille** (Clinchant et al., 2024) encapsula ColBERT em uma API de alto nível, compatível com LangChain e LlamaIndex:

```python
from ragatouille import RAGPretrainedModel
RAG = RAGPretrainedModel.from_pretrained("colbert-ir/colbertv2.0")
RAG.index(collection=docs, index_name="juridico")
results = RAG.search(query="habeas corpus", k=5)
```

### 3.6 Ficha Técnica #T20

| Campo | Conteúdo |
|---|---|
| **ID** | #T20 — ColBERT / Late Interaction |
| **Categoria** | Complementar |
| **Stack** | RAGatouille + ColBERTv2 + PLAID |
| **Vantagem principal** | Precisão próxima a cross-encoders com velocidade de bi-encoders |
| **Limitação principal** | Índice 3–5× maior; requer infraestrutura específica |
| **Referência** | Khattab & Zaharia, SIGIR 2020 |

---
## 4. Time-Aware RAG — Relevância Temporal

### 4.1 Por que a Data Importa no Domínio Jurídico

No Direito, **regulações são revogadas, súmulas são canceladas, leis são alteradas**. Um sistema RAG sem consciência temporal pode retornar:

- Jurisprudência anterior a decisão vinculante do STF
- Portaria revogada como se ainda estivesse vigente
- Texto de lei antes de emenda constitucional

Isso configura **risco jurídico grave** em aplicações de assessoria.

### 4.2 Decay Function — Fórmula de Penalização Temporal

O OpenSearch implementa **function score queries** com funções de decay. A mais usada é a **exponencial**:

```
decay_score(data_doc) = exp(-λ × max(0, age_days - offset))

Onde:
  λ (lambda)  = ln(2) / scale  → controla velocidade do decay
  scale       = half-life em dias (ex: 365 = 1 ano)
  offset      = período de graça sem penalização (ex: 30 dias)
  age_days    = (data_atual - data_documento) em dias
```

**Exemplo prático:**

| Documento | Idade | Decay Score (scale=365) |
|---|---|---|
| Lei publicada ontem | 1 dia | 0.998 |
| Lei de 1 ano atrás | 365 dias | 0.500 |
| Lei de 2 anos atrás | 730 dias | 0.250 |
| Lei de 5 anos atrás | 1825 dias | 0.031 |

### 4.3 Configuração no OpenSearch

```json
{
  "query": {
    "function_score": {
      "query": { "match": { "content": "habeas corpus" } },
      "functions": [{
        "exp": {
          "data_publicacao": {
            "origin": "now",
            "scale": "365d",
            "offset": "30d",
            "decay": 0.5
          }
        }
      }],
      "score_mode": "multiply",
      "boost_mode": "multiply"
    }
  }
}
```

### 4.4 Estratégias de Design Temporal

| Estratégia | Quando usar | Configuração |
|---|---|---|
| Decay suave (scale=730d) | Jurisprudência histórica relevante | Preserva docs antigos com peso |
| Decay agressivo (scale=180d) | Regulações operacionais | Prioriza forte documentos recentes |
| Hard filter | Legislação vigente apenas | `filter: { range: { data: { gte: "now-1y" } } }` |
| Boost por versão | Controle de versão de normas | Campo `versao_vigente: true` com boost=3 |

### 4.5 Ficha Técnica — Time-Aware RAG

| Campo | Conteúdo |
|---|---|
| **Técnica** | Time-Aware RAG |
| **Categoria** | Complementar |
| **Stack** | OpenSearch function_score + campo date |
| **Risco mitigado** | Retorno de normas revogadas ou desatualizadas |
| **Parâmetro-chave** | `scale` (half-life do decay) — calibrar por domínio |
| **Referência** | OpenSearch Documentation — Function Score Queries |

---
## 5. DSPy — Otimização Automática de Pipelines RAG

### 5.1 O Problema do Prompt Engineering Manual

Pipelines RAG dependem criticamente de prompts bem elaborados. O problema:

- Prompts são frágeis: pequenas mudanças de modelo ou domínio os degradam
- Engenharia manual é custosa e não reproduzível
- Não existe métrica objetiva para "qual prompt é melhor"
- Few-shot examples escolhidos manualmente são subótimos

### 5.2 DSPy — Programação Declarativa de LLMs

O **DSPy** (Stanford NLP, 2024) resolve isso tratando pipelines RAG como **programas compiláveis**:

```
Desenvolvedor define:              Compilador DSPy produz:
┌─────────────────────┐           ┌──────────────────────────────┐
│ Módulos:            │           │ Prompt otimizado:            │
│  - Retrieve(k=5)    │  ──────►  │ "Dado o contexto jurídico    │
│  - ChainOfThought   │   compila │  abaixo, analise step-by-    │
│                     │           │  step: [contexto]            │
│ Métrica:            │           │  Exemplos: [3 exemplos       │
│  - Faithfulness     │           │  few-shot otimizados]        │
│  - Answer Accuracy  │           │  Pergunta: {question}"      │
│                     │           │                              │
│ Dataset:            │           │ Few-shot examples:           │
│  50 pares Q&A       │           │  Selecionados por bootstrap  │
└─────────────────────┘           └──────────────────────────────┘
```

### 5.3 Módulos DSPy

| Módulo | Descrição | Uso |
|---|---|---|
| `dspy.Predict` | Geração simples sem raciocínio explícito | Classificação, extração |
| `dspy.ChainOfThought` | Raciocínio passo a passo (CoT) | Q&A complexas, análise jurídica |
| `dspy.ReAct` | Raciocínio + Ação (tool use) | Agentes que usam ferramentas |
| `dspy.Retrieve` | Busca em índice vetorial | Integração com RAG |
| `dspy.MultiChainComparison` | Compara múltiplas cadeias | Verificação de consistência |

### 5.4 Otimizadores (Compiladores)

| Otimizador | Descrição | Custo |
|---|---|---|
| `BootstrapFewShot` | Gera exemplos few-shot por bootstrapping | Baixo |
| `BootstrapFewShotWithRandomSearch` | Adiciona busca aleatória de hiperparâmetros | Médio |
| `MIPROv2` | Otimização bayesiana de prompts e exemplos | Alto |
| `COPRO` | Otimiza apenas instruções do prompt (sem exemplos) | Médio |

### 5.5 Exemplo de Antes e Depois da Compilação

**Antes (prompt manual):**
```
Responda a pergunta jurídica com base no contexto fornecido.
Contexto: {context}
Pergunta: {question}
Resposta:
```

**Depois (DSPy BootstrapFewShot compilado):**
```
Você é um especialista em Direito Penal brasileiro. Analise o contexto
jurídico fornecido e responda à pergunta identificando:
1) O fundamento legal aplicável
2) A jurisprudência relevante mencionada
3) A conclusão objetiva

---
Contexto: [Art. 157 CP: Subtrair coisa móvel alheia...]
Pergunta: Qual o tipo penal do réu que subtraiu celular com ameaça?
Raciocínio: O Art. 157 do CP tipifica o roubo como subtração com violência
ou grave ameaça. Celular é coisa móvel alheia. A ameaça configura o
elemento normativo. Logo: roubo simples (Art. 157, caput).
Resposta: Roubo simples, Art. 157, caput, do Código Penal.
---

[2 exemplos adicionais few-shot...]

Contexto: {context}
Pergunta: {question}
```

**Melhoria típica:** +15–30% em Faithfulness, +10–20% em Answer Accuracy.

### 5.6 Ficha Técnica #T25

| Campo | Conteúdo |
|---|---|
| **ID** | #T25 — DSPy |
| **Categoria** | Complementar |
| **Stack** | DSPy + RAGAS (métricas) + qualquer LLM |
| **Vantagem principal** | Elimina engenharia manual de prompts com melhoria mensurável |
| **Limitação principal** | Curva de aprendizado significativa; compute intensivo na compilação |
| **Referência** | Khattab et al., ICLR 2024 |

---
## 6. Guia de Decisão — Qual Técnica Usar no Projeto Final?

```
Meu corpus tem imagens ou tabelas importantes?
  SIM ──► Multimodal RAG (Docling + CLIP + OpenSearch)
  NÃO ──► continue ▼

Tenho custo de tokens crítico ou contexto limitado?
  SIM ──► Compression RAG (LLMLingua antes de enviar ao LLM)
  NÃO ──► continue ▼

Preciso de alta precisão em queries técnicas específicas?
  SIM ──► ColBERT / RAGatouille (late interaction MaxSim)
  NÃO ──► continue ▼

Meu domínio tem documentos com validade temporal (leis, portarias)?
  SIM ──► Time-Aware RAG (OpenSearch decay exponencial)
  NÃO ──► continue ▼

Quero otimizar automaticamente meu pipeline existente?
  SIM ──► DSPy (BootstrapFewShot ou MIPROv2)
  NÃO ──► Seu pipeline canônico das Aulas 1-10 já é suficiente
```

**Nota:** As técnicas são complementares. Um pipeline avançado pode combinar:
**Time-Aware + Hybrid Search + ColBERT Reranker + LLMLingua + DSPy**

---
## 7. Stack Tecnológico desta Aula

| Biblioteca | Versão | Função |
|---|---|---|
| `docling` | ≥2.0 | Extração multimodal de PDFs |
| `transformers` (CLIP) | ≥4.40 | Embeddings de imagem |
| `llmlingua` | ≥0.2 | Compressão de contexto |
| `ragatouille` | ≥0.0.8 | Interface ColBERT simplificada |
| `dspy-ai` | ≥2.4 | Otimização automática de pipeline |
| `opensearch-py` | ≥2.4 | Cliente OpenSearch (time-aware) |
| `langfuse` | ≥2.0 | Observabilidade e métricas |
| `ragas` | ≥0.1 | Avaliação de pipeline RAG |

---

## 8. Critérios de Avaliação

| Critério | Peso | Descrição |
|---|---|---|
| Lab Multimodal funcional | 20% | Query recupera texto + imagem simultaneamente |
| LLMLingua com análise | 20% | Antes/depois documentados com taxa de compressão |
| ColBERT benchmarked | 20% | nDCG@10 comparado com BGE-M3 |
| Time-Aware configurado | 20% | Decay exponencial funcionando no OpenSearch |
| DSPy compilado | 20% | Prompt antes/depois com métrica RAGAS melhorada |

---

*Próximos: Labs práticos nas células abaixo e em aula11/labs/*